In [24]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers
import time  # For sleep/timing between motor commands

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()
LastColor = None
got.mechanical_joint_control(angle1=0, angle2=90, angle3=-90, duration=500)
got.mechanical_clamp_close()

def AP_centralization_approaching(distance=0.17225, gap=12.5, fwd_spd=25, strafe_spd=20):
    
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """

    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()

    if not AP_info:
        print("No Tag Detected")
        return

    while True:
        got.mechanical_joint_control(angle1=0, angle2=90, angle3=-120, duration=500)
        # Refresh tag data every iteration for responsive corrections.
        AP_info = got.get_apriltag_total_info()

        if not AP_info:
            got.mecanum_stop()
            break

        AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
        AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

        if AP_x < 320 - gap:
            # Tag is to the LEFT of center — strafe left to re-align.
            # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            # Tag is to the RIGHT of center — strafe right to re-align.
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            # Tag is centered but still too far — drive straight forward.
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            # Tag is centered AND within target distance — stop and exit.
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break
        time.sleep(0.1)

try:
    while True:
        # get_color_total_info() returns a list of detected color results.
        # Index [0] gives the name of the most prominent color (e.g. "Red", "Green").
        color = got.get_color_total_info()[0]

        # Update the robot's display based on detected color.

        if color == "Red":
            got.screen_display_background(3) # Red background
            LastColor = "Red"

        elif color == "Green":
            got.screen_display_background(6) # Green background
            LastColor = "Green"
            
        else:
            got.screen_display_background(0) # Black background

        # Print current color, or a message if nothing is detected.
        if color:
            print(color)
        else:
            print("No color!")

        # Clear console output so each refresh shows only the latest color.
        clear_output(wait=True)

        while LastColor == "Green":
                tag = got.get_apriltag_total_info()
                if tag:
                    AP_centralization_approaching(distance=0.2725, gap=20, fwd_spd=25, strafe_spd=20)
                    got.mechanical_clamp_release()
                    got.mechanical_joint_control(angle1=5, angle2=0, angle3=-90, duration=500)
                    time.sleep(1)
                    got.mechanical_clamp_close()
                    time.sleep(0.5)
                    got.mechanical_joint_control(angle1=0, angle2=90, angle3=-90, duration=500)
                    break
                else:
                    print("No Tag Visible")
                    break

except KeyboardInterrupt:
    # Safely stop the robot's wheels if the cell is interrupted.
    got.mecanum_stop()
    print("Done")

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "WSAGetOverlappedResult: Connection timed out (A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond.
 -- 10060)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"WSAGetOverlappedResult: Connection timed out (A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond.\r\n -- 10060)"}"
>